# CPTR 480 — Week 1 Wednesday: Board Bring-Up Methodology
### April 1, 2026

**Today's agenda**
1. Lab 1 debrief — quick check-in (5 min)
2. The bring-up pyramid — a systematic methodology (10 min)
3. Live KiCad walkthrough — SDR & 2026 board circuit descriptions (40 min)
4. Preview of Lab 2 (5 min)


---
## Part 1 — Lab 1 Debrief (5 min)

**Quick show of hands:**
- Who finished all 5 parts?
- Where did people get stuck? (1–2 volunteers)

> **Key question to surface:** *"What does an empty I2C scan result mean, and how would you start debugging it?"*
— Hold that thought; the bring-up pyramid answers it.


---
## Part 2 — The Bring-Up Pyramid (10 min)

Every piece of hardware you ever touch follows the same debug order. Skipping layers wastes hours.

```
        ┌─────────────┐
        │ Application │   ← Only once everything below works
        ├─────────────┤
        │Communication│   ← I2C scan, SPI loopback
        ├─────────────┤
        │   Clocks    │   ← ICs won't respond without supply clock
        ├─────────────┤
        │ Power Rails │   ← Wrong voltage can destroy hardware
        ├─────────────┤
        │Visual Insp. │   ← Bridges, orientation, missing parts
        └─────────────┘
```

### Layer 1 — Visual Inspection
Before **any** power is applied: check orientation, solder bridges, missing decoupling caps. Use a loupe or phone camera.

### Layer 2 — Power Rails
| Rail | Expected | Why it matters |
|------|----------|----------------|
| VBUS | 5.0 V | USB power in |
| 3V3  | 3.3 V | Logic supply for Pico, OLED, Si5351a |
| AVDD | 3.3 V (analog) | PCM1808 analog supply — noise-sensitive |

### Layer 3 — Clocks
Many ICs **will not respond on I2C/SPI** unless their clock is running. The Si5351A needs I2C config before it outputs anything. The PCM1808 needs MCLK before its I2C address is visible.

### Layer 4 — Communication
```python
import machine
i2c = machine.I2C(0, sda=machine.Pin(4), scl=machine.Pin(5))
print("Devices found:", [hex(a) for a in i2c.scan()])
# Si5351A → 0x60,  OLED → 0x3C
```

### Layer 5 — Application
Only here do you run your full program.

> **Tie-back to debrief:** *"Empty i2c.scan() — which layer failed?"*


---
## Part 3 — Live KiCad Walkthrough: 2026 Board Schematic (40 min)

> *[Switch to KiCad. Open `Intro-to-CAD-2026/Intro-to-CAD-2026.kicad_sch` on the projector.]*

---

### 0. KiCad orientation for CS students (5 min)

*For EE students this is review — ask them to spot anything unusual on their own board.*

**Schematic conventions to explain:**
- **Wire** = electrical connection; crossing without a dot = no connection
- **Net label** = a name shared across a sheet; same name = same wire even if not drawn connected
- **Power symbol** (`+3V3`, `GND`, `AGND`) = implicit global connection to every matching symbol
- **Pin numbers vs net names** — the net name (`I2C_SDA`) is what matters in firmware, not the physical pin number
- **Reference designator** — `R12`, `C4`, `U2` — how you find a part in the BOM and on the PCB

**Navigating KiCad:**
- Scroll to zoom; middle-click drag to pan
- `Ctrl+F` — find a net label or reference anywhere in the project
- Hierarchy Navigator (`Tools → Navigate Schematic Hierarchy`) — shows all sub-sheets
- Click a hierarchical sheet block → opens that sub-sheet

---

### 1. Top-level sheet — system architecture (5 min)

*[Show the top-level sheet. Point out the hierarchical sheet blocks.]*

Sub-sheets to walk through:
| Sheet | What it contains |
|-------|-----------------|
| `raspberry_pi_pico` | YD-RP2040, GPIO assignments, USB |
| `si5351a` | Programmable clock generator (local oscillator) |
| `pcm1808` | 24-bit stereo audio ADC |
| `qsd-sdr` | Quadrature Sampling Detector — the RF mixer |
| `oled` | SSD1306 128×64 I2C display |

> *"This board is a software-defined radio receiver front end plus a display. The Pico controls everything and does the DSP in software."*

---

### 2. Pico / YD-RP2040 sheet (5 min)

*[Open `raspberry_pi_pico.kicad_sch`]*

- Point out the GPIO net labels leaving the sheet — these are what tie everything together
- Show `I2C_SDA` / `I2C_SCL` — used by OLED **and** Si5351a on the same bus
- Show `MCLK` — driven by the Pico's PIO at 24.576 MHz; the PCM1808 **cannot work** without it
- USB D+/D− — how the Pico appears as a USB CDC serial device on the host

---

### 3. Si5351A — Programmable Clock Generator (5 min)

*[Open `si5351a.kicad_sch`]*

**What it does:** Takes a 24.576 MHz crystal reference and synthesizes precise output frequencies via I2C-programmable PLLs.

*Key things to point out:*
- Crystal + load caps (C_L) — why the value matters (frequency pulling)
- Decoupling caps on VDD — must be close to IC on the PCB to work
- `I2C_SDA` / `I2C_SCL` net labels linking back to the Pico
- Address pin — determines I2C address (`0x60`)
- Output clock pins → where do they go? Follow the net to the QSD sheet

---

### 4. QSD-SDR — Quadrature Sampling Detector (10 min)

*[Open `qsd-sdr.kicad_sch`]*

**Concept (1 min):** A QSD mixes an incoming RF signal with two local oscillator clocks that are 90° apart (I and Q channels). The math works out so that software can distinguish signals above and below the LO frequency — this is how you tune a software-defined radio.

*Key things to point out:*
- RF input path — antenna → (any filter) → mixer switch inputs
- The 4-phase clock signals from the Si5351A (CLK0 = 0°, CLK1 = 90°, CLK2 = 180°, CLK3 = 270°)
- I and Q sample-and-hold capacitors
- Output op-amp buffers → to ADC inputs
- `AGND` vs `GND` — analog and digital grounds, connected at one star point

> *"This is why AGND exists — high-frequency digital switching noise from the Pico must not ride back through the same ground plane as the sensitive 100 mV RF signal."*

---

### 5. PCM1808 — Audio ADC (5 min)

*[Open `pcm1808.kicad_sch`]*

**What it does:** Digitizes the I and Q analog outputs from the QSD at 24-bit resolution and streams them to the Pico over I2S.

*Key things to point out:*
- MCLK / LRCK / BICK / DOUT — the four I2S lines (also called I2S clocks + data)
- MCLK comes from the Pico PIO — without it the chip is silent and won't appear on the I2C bus
- AVDD/AGND supply filtering — separate from digital supply
- Differential input pins (IN1L+/IN1L−) — show how the I/Q signals connect from the QSD

---

### 6. OLED sheet (2 min)

*[Open `oled.kicad_sch`]*

- SSD1306 controller, I2C address `0x3C`
- Same `I2C_SDA`/`I2C_SCL` bus as the Si5351A
- `RES` reset line — must be toggled high before issuing commands

---

### Wrap-up (3 min)

> *"EE students: anything look different from what you'd expect? Anyone spot something they'd do differently?"*

**HW 1 reminder:** You now have enough context to fill in the GPIO pin table for HW 1. Open the schematic tonight, trace each peripheral, fill in the table. Due **Monday April 6 at 11:00 a.m.**


---
## Part 4 — Preview: Lab 2 (5 min)

**Lab 2 (Tuesday, April 7):** Pico Debug Stack Setup + Full Board Validation + README

Two things to have ready **before you walk in:**

### 1. Your HW 1 README skeleton
Lab 2 asks you to fill in the GPIO pin table for every peripheral. If you arrive without it, you'll spend 45 minutes writing Markdown tables instead of doing hardware work.

**Tonight:** Open the schematic (we just walked through it), trace each peripheral's GPIO connections, fill in:

| Peripheral | GPIO | Direction | Protocol |
|------------|------|-----------|----------|
| OLED SDA   | ?    | Out       | I2C      |
| OLED SCL   | ?    | Out       | I2C      |
| Si5351a SDA| ?    | Out       | I2C      |
| PCM1808 MCLK| ?   | Out       | PIO/Clock|
| ...        | ...  | ...       | ...      |

### 2. Your C toolchain must work
Lab 2 validates the **C/C++ Pico SDK** toolchain. Build the `blink` example from the Pico SDK before Tuesday. If it compiles and flashes, you're ready. If not, come to office hours Thursday.

---

## Summary

| Layer | Check | Pass condition |
|-------|-------|----------------|
| Visual | Eyes + loupe | No bridges, correct orientation |
| Power | Multimeter | 3.3 V on 3V3, 5 V on VBUS |
| Clocks | Scope / logic analyzer | MCLK toggling at 24.576 MHz |
| Communication | `i2c.scan()` | 0x3C (OLED), 0x60 (Si5351a) visible |
| Application | Your code | Correct behavior |

**HW 1 due Monday April 6 at 11:00 a.m.**
